# Задание 2: Построение и оценка качества бейзлайна

## Тип задачи
**Регрессия**: предсказание рейтинга шоколадной плитки (от 1 до 5)

## Метрика качества

**Метрика:** RMSE

Выбрана RMSE потому что:
- Имеет ту же размерность, что и целевая переменная (рейтинг)
- Сильнее штрафует большие ошибки по сравнению с MAE, что важно для предсказания рейтинга
- Стандартная метрика для регрессионных задач


## Импорт библиотек

In [98]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# Фиксируем random_state для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Загрузка и предобработка данных

In [99]:
# Загрузка данных
url = ('https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2022/2022-01-18/chocolate.csv')
df = pd.read_csv(url)

print(f"Размер датасета: {df.shape}")
print(f"\nПервые 5 строк:")
print(df.head())

print(f"\nИнформация о данных:")
print(df.info())


Размер датасета: (2530, 10)

Первые 5 строк:
    ref company_manufacturer company_location  review_date  \
0  2454                 5150           U.S.A.         2019   
1  2458                 5150           U.S.A.         2019   
2  2454                 5150           U.S.A.         2019   
3  2542                 5150           U.S.A.         2021   
4  2546                 5150           U.S.A.         2021   

  country_of_bean_origin specific_bean_origin_or_bar_name cocoa_percent  \
0               Tanzania            Kokoa Kamili, batch 1           76%   
1     Dominican Republic                  Zorzal, batch 1           76%   
2             Madagascar           Bejofo Estate, batch 1           76%   
3                   Fiji            Matasawalevu, batch 1           68%   
4              Venezuela            Sur del Lago, batch 1           72%   

  ingredients     most_memorable_characteristics  rating  
0    3- B,S,C          rich cocoa, fatty, bready    3.25  
1    3- B,S,C

## Предобработка признаков

### Обработка целевой переменной и числовых признаков
- `rating` - целевая переменная (рейтинг от 1 до 5)
- `cocoa_percent` - процент какао (преобразуем из строки в число)
- `review_date` - год отзыва

### Обработка категориальных признаков

Используем **Label Encoding** для категориальных переменных.

**Почему Label Encoding:**
- Простой и быстрый метод кодирования
- Не увеличивает размерность признакового пространства (в отличие от One-Hot Encoding)
- Подходит для бейзлайн-моделей (линейная регрессия, дерево решений)
- One-Hot Encoding создал бы слишком много признаков для данной задачи

Категориальные признаки: `country_of_bean_origin`, `company_manufacturer`, `company_location`


In [100]:
# Копируем данные для работы
df_processed = df.copy()

# Преобразование cocoa_percent из строки в число
df_processed['cocoa_percent'] = (
    df_processed['cocoa_percent'].str.rstrip('%').astype(float)
)

# Заполнение пропусков в ingredients (если есть)
df_processed['ingredients'] = df_processed['ingredients'].fillna('Unknown')

# Проверка пропусков в целевой переменной
print(f"Пропуски в rating: {df_processed['rating'].isnull().sum()}")
print(f"Пропуски в cocoa_percent: {df_processed['cocoa_percent'].isnull().sum()}")

# Проверка распределения целевой переменной
print(f"\nСтатистика по rating:")
print(df_processed['rating'].describe())


Пропуски в rating: 0
Пропуски в cocoa_percent: 0

Статистика по rating:
count    2530.000000
mean        3.196344
std         0.445321
min         1.000000
25%         3.000000
50%         3.250000
75%         3.500000
max         4.000000
Name: rating, dtype: float64


In [101]:
# Подготовка признаков и целевой переменной
target = df_processed['rating'].copy()

# Числовые признаки
numeric_features = ['cocoa_percent', 'review_date']

# Категориальные признаки для кодирования
categorical_features = [
    'country_of_bean_origin',
    'company_manufacturer',
    'company_location'
]

# Создание признаков
X_numeric = df_processed[numeric_features].copy()

# Label Encoding для категориальных признаков
label_encoders = {}
X_categorical = pd.DataFrame()

for feature in categorical_features:
    le = LabelEncoder()
    # Заполняем пропуски перед кодированием
    df_processed[feature] = df_processed[feature].fillna('Unknown')
    X_categorical[feature] = le.fit_transform(df_processed[feature])
    label_encoders[feature] = le

# Объединяем все признаки
X = pd.concat([X_numeric, X_categorical], axis=1)

print(f"Размерность признакового пространства: {X.shape}")
print(f"\nПризнаки:")
print(X.columns.tolist())
print(f"\nПервые 5 строк признаков:")
print(X.head())


Размерность признакового пространства: (2530, 5)

Признаки:
['cocoa_percent', 'review_date', 'country_of_bean_origin', 'company_manufacturer', 'company_location']

Первые 5 строк признаков:
   cocoa_percent  review_date  country_of_bean_origin  company_manufacturer  \
0           76.0         2019                      52                     0   
1           76.0         2019                      13                     0   
2           76.0         2019                      28                     0   
3           68.0         2021                      16                     0   
4           72.0         2021                      60                     0   

   company_location  
0                62  
1                62  
2                62  
3                62  
4                62  


## Разбиение на тренировочную и тестовую выборки

Разделяем датасет в соотношении 80/20 (train/test).

**Почему 80/20:**
- 80% данных достаточно для обучения моделей
- 20% данных достаточно для надежной оценки качества
- Стандартное соотношение в машинном обучении

`random_state=42` фиксируем для воспроизводимости результатов.


In [102]:
# Разбиение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
)

train_size_pct = X_train.shape[0] / len(X) * 100
test_size_pct = X_test.shape[0] / len(X) * 100

print(f"Размер тренировочной выборки: {X_train.shape[0]} "
      f"({train_size_pct:.1f}%)")
print(f"Размер тестовой выборки: {X_test.shape[0]} ({test_size_pct:.1f}%)")
print(f"\nСтатистика по y_train:")
print(y_train.describe())
print(f"\nСтатистика по y_test:")
print(y_test.describe())


Размер тренировочной выборки: 2024 (80.0%)
Размер тестовой выборки: 506 (20.0%)

Статистика по y_train:
count    2024.000000
mean        3.202569
std         0.442475
min         1.000000
25%         3.000000
50%         3.250000
75%         3.500000
max         4.000000
Name: rating, dtype: float64

Статистика по y_test:
count    506.000000
mean       3.171443
std        0.456124
min        1.000000
25%        2.750000
50%        3.250000
75%        3.500000
max        4.000000
Name: rating, dtype: float64


## Baseline

Самый простой вариант — предсказывать константу (среднее или медиану). Это минимальный уровень, который должна превзойти модель.


In [103]:
# Функция для вычисления RMSE
def calculate_rmse(y_true, y_pred):
    """
    Вычисляет Root Mean Squared Error.
    
    Args:
        y_true: Истинные значения целевой переменной
        y_pred: Предсказанные значения
        
    Returns:
        float: Значение RMSE
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Константное предсказание: среднее значение
constant_mean = y_train.mean()
y_pred_mean = np.full(len(y_test), constant_mean)
rmse_mean = calculate_rmse(y_test, y_pred_mean)

print(f"Константное предсказание (среднее):")
print(f"  Значение: {constant_mean:.4f}")
print(f"  RMSE на тестовой выборке: {rmse_mean:.4f}")

# Константное предсказание: медиана
constant_median = y_train.median()
y_pred_median = np.full(len(y_test), constant_median)
rmse_median = calculate_rmse(y_test, y_pred_median)

print(f"\nКонстантное предсказание (медиана):")
print(f"  Значение: {constant_median:.4f}")
print(f"  RMSE на тестовой выборке: {rmse_median:.4f}")

# Сохраняем лучшее константное предсказание для сравнения
baseline_rmse = min(rmse_mean, rmse_median)
print(f"\nЛучшее константное предсказание (baseline): RMSE = {baseline_rmse:.4f}")


Константное предсказание (среднее):
  Значение: 3.2026
  RMSE на тестовой выборке: 0.4567

Константное предсказание (медиана):
  Значение: 3.2500
  RMSE на тестовой выборке: 0.4624

Лучшее константное предсказание (baseline): RMSE = 0.4567


## Обучение бейзлайн-модели

Обучаем две простые модели для сравнения.

**Линейная регрессия:**
- Простая и интерпретируемая модель
- Быстро обучается
- Требует масштабирования признаков (StandardScaler), так как чувствительна к масштабу
- Классический бейзлайн для регрессионных задач

**Дерево решений:**
- Не требует масштабирования признаков
- Может уловить нелинейные зависимости
- Ограничиваем `max_depth=5` для предотвращения переобучения
- Альтернативный бейзлайн для сравнения с линейной моделью


In [104]:
# Масштабирование признаков для линейной регрессии
# StandardScaler приводит признаки к среднему=0, std=1
# Необходимо, так как линейная регрессия чувствительна к масштабу признаков
# и у нас смешаны числовые признаки с разными масштабами (cocoa_percent 0-100, review_date 2000+, категориальные 0-N)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Признаки масштабированы с помощью StandardScaler")
print(f"Среднее значение признаков (train): {X_train_scaled.mean(axis=0)[:3]}")
print(f"Стандартное отклонение (train): {X_train_scaled.std(axis=0)[:3]}")


Признаки масштабированы с помощью StandardScaler
Среднее значение признаков (train): [8.95638416e-16 7.49159189e-15 4.30046863e-17]
Стандартное отклонение (train): [1. 1. 1.]


In [105]:
# Обучение линейной регрессии
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

# Предсказания на тренировочной выборке
y_train_pred = linear_model.predict(X_train_scaled)
rmse_train = calculate_rmse(y_train, y_train_pred)

print("Линейная регрессия обучена")
print(f"RMSE на тренировочной выборке: {rmse_train:.4f}")
print(f"R² (коэффициент детерминации): {linear_model.score(X_train_scaled, y_train):.4f}")


Линейная регрессия обучена
RMSE на тренировочной выборке: 0.4314
R² (коэффициент детерминации): 0.0491


## Оценка качества на тестовой выборке

Оценка качества модели на отложенной тестовой выборке.


In [106]:
# Предсказания на тестовой выборке
y_test_pred = linear_model.predict(X_test_scaled)
rmse_test = calculate_rmse(y_test, y_test_pred)

print("Оценка качества на тестовой выборке:")
print(f"RMSE на тестовой выборке: {rmse_test:.4f}")
r2_score = linear_model.score(X_test_scaled, y_test)
print(f"R² (коэффициент детерминации): {r2_score:.4f}")

# Сравнение с baseline
improvement = ((baseline_rmse - rmse_test) / baseline_rmse * 100)
print(f"\nСравнение с константным предсказанием:")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Linear Regression RMSE: {rmse_test:.4f}")
print(f"Улучшение: {improvement:.2f}%")


Оценка качества на тестовой выборке:
RMSE на тестовой выборке: 0.4530
R² (коэффициент детерминации): 0.0117

Сравнение с константным предсказанием:
Baseline RMSE: 0.4567
Linear Regression RMSE: 0.4530
Улучшение: 0.82%


## Дерево решений

Дерево решений как альтернативный бейзлайн для сравнения с линейной регрессией.

**Особенности:**
- Не требует масштабирования признаков (в отличие от линейной регрессии)
- Может уловить нелинейные зависимости между признаками и целевой переменной
- `max_depth=5` — ограничение глубины для предотвращения переобучения (слишком глубокое дерево запомнит тренировочные данные)


In [107]:
# Обучение дерева решений (не требует масштабирования)
tree_model = DecisionTreeRegressor(
    max_depth=5,
    random_state=RANDOM_STATE
)
tree_model.fit(X_train, y_train)

# Предсказания на тестовой выборке
y_test_pred_tree = tree_model.predict(X_test)
rmse_test_tree = calculate_rmse(y_test, y_test_pred_tree)

print("Дерево решений обучено")
print(f"RMSE на тестовой выборке: {rmse_test_tree:.4f}")
tree_r2_score = tree_model.score(X_test, y_test)
print(f"R² (коэффициент детерминации): {tree_r2_score:.4f}")

# Сравнение моделей
print(f"\nСравнение моделей:")
print(f"Baseline (константа): {baseline_rmse:.4f}")
print(f"Linear Regression: {rmse_test:.4f}")
print(f"Decision Tree (max_depth=5): {rmse_test_tree:.4f}")

# Выбираем лучшую модель
best_model_name = "Linear Regression"
best_rmse = rmse_test
if rmse_test_tree < rmse_test:
    best_model_name = "Decision Tree"
    best_rmse = rmse_test_tree

print(f"\nЛучшая модель: {best_model_name} (RMSE = {best_rmse:.4f})")


Дерево решений обучено
RMSE на тестовой выборке: 0.4506
R² (коэффициент детерминации): 0.0221

Сравнение моделей:
Baseline (константа): 0.4567
Linear Regression: 0.4530
Decision Tree (max_depth=5): 0.4506

Лучшая модель: Decision Tree (RMSE = 0.4506)


## Выводы

**Результаты:**
- Датасет разделен 80/20, `random_state=42` для воспроизводимости
- Baseline: RMSE = 0.4567
- Линейная регрессия: RMSE = 0.4530 (улучшение 0.82%)
- Дерево решений: RMSE = 0.4506

**Что использовали:**
- Label Encoding — простой способ кодирования категорий
- StandardScaler — нормализация для линейной регрессии
- Линейная регрессия и дерево решений как базовые модели
